# Brain MRI 3D inference — SwinUNETR (Colab)

Runs one case through the project's own `load_input_volume` preprocessing and
`predict_model_card_case` inference/postprocessing. It does not duplicate those
steps. The exact checkpoint-compatible releases recorded by this project are
**PyTorch 2.14.1** and **MONAI 1.6.1**. The project uses a 64³ runtime ROI
(the model-card ROI is 128³), sliding-window batch size 1, overlap 0.7,
sigmoid probabilities, and threshold 0.5.

This is research software, not a diagnosis or treatment recommendation.
Clinical performance is unvalidated. A run is not a clinical validation.

## Before starting

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU).
   The code automatically chooses CUDA when available and otherwise uses CPU.
2. Upload `capstone_colab_project_source.zip` from alongside this notebook when
   prompted. This archive is a snapshot of the actual project source used to
   build this notebook, including the current `monai_engine.py` and `brats.py`.
3. Place the existing `best.pth` checkpoint and four co-registered, skull-stripped
   NIfTI files in Google Drive. Name them `<CASE_ID>-t1n`, `-t1c`, `-t2w`, and
   `-t2f`, with either `.nii` or `.nii.gz` suffixes.
   Do not substitute, reorient, or resample modalities here.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
bundle_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(bundle_names) != 1:
    raise ValueError("Upload exactly one capstone_colab_project_source.zip file.")

PROJECT_ROOT = Path("/content/capstone_mri_project")
if PROJECT_ROOT.exists():
    raise FileExistsError(f"Refusing to overwrite existing project folder: {PROJECT_ROOT}")
with zipfile.ZipFile(bundle_names[0]) as archive:
    archive.extractall(PROJECT_ROOT)

required = [
    PROJECT_ROOT / "src/segmentation3d/monai_engine.py",
    PROJECT_ROOT / "src/data/brats.py",
    PROJECT_ROOT / "integrated_pipeline/model_registry.json",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("The uploaded project snapshot is incomplete: " + ", ".join(missing))
print(f"Loaded project source snapshot from {bundle_names[0]}")


## Install the verified runtime

The checkpoint was strictly loaded using PyTorch 2.14.1+cpu and MONAI 1.6.1.
Install that exact PyTorch release and MONAI version; the PyTorch wheel selected
by pip can use Colab CUDA when available. If Colab requests a runtime restart,
restart the session and continue from the project source cell.


In [ ]:
%pip install -q "torch==2.14.1" "monai==1.6.1" "einops>=0.7" "nibabel>=5" "scipy>=1.10" "matplotlib>=3.7" "numpy>=1.24" "PyYAML>=6" "Pillow>=10" "psutil>=5.9"


In [ ]:
import sys
import torch
import monai

assert torch.__version__.split("+", 1)[0] == "2.14.1", (
    f"Expected PyTorch 2.14.1, found {torch.__version__}. Restart the Colab runtime "
    "if prompted, then rerun the install cell."
)
assert monai.__version__ == "1.6.1", f"Expected MONAI 1.6.1, found {monai.__version__}."

sys.path.insert(0, str(PROJECT_ROOT))
print(f"PyTorch: {torch.__version__}")
print(f"MONAI: {monai.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU fallback; runtime may be substantially slower.")


## Select the case and checkpoint

Mount Drive and edit the three paths and case ID below. The modality mapping is
fixed to **T1, T1ce, T2, FLAIR** (`t1n`, `t1c`, `t2w`, `t2f`). The loader rejects
missing modalities, non-3D images, shape/affine disagreement, non-finite data,
or all-zero scans. Its channel-wise nonzero normalization and native-axis
geometry preservation are used directly.

The existing 3D checkpoint is at `2_3d_segmentation/outputs/segmentation3d/SwinUNETR/best.pth` in the local project. Copy that file to the Drive path configured below; Colab cannot read the Windows `C:\\Captone\\...` path directly. The 2D segmentation and classification checkpoints are for the local Streamlit app and are not used in this 3D notebook.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CASE_ID = "replace-with-case-id"
INPUT_DIR = Path("/content/drive/MyDrive/Capstone_MRI_Project/input_case")
CHECKPOINT_PATH = Path("/content/drive/MyDrive/Capstone_MRI_Project/best.pth")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Capstone_MRI_Project/colab_outputs") / CASE_ID

MODALITIES = ("t1n", "t1c", "t2w", "t2f")
SOURCE_FILES = {}
for name in MODALITIES:
    candidates = [
        INPUT_DIR / f"{CASE_ID}-{name}.nii.gz",
        INPUT_DIR / f"{CASE_ID}-{name}.nii",
    ]
    matches = [path for path in candidates if path.is_file()]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous duplicate NIfTI inputs for {name}: {matches}")
    SOURCE_FILES[name] = matches[0] if matches else candidates[0]
missing_files = [str(path) for path in [*SOURCE_FILES.values(), CHECKPOINT_PATH] if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Required input/checkpoint file(s) missing: " + ", ".join(missing_files))
if not CASE_ID or any(character not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for character in CASE_ID):
    raise ValueError("CASE_ID must contain only letters, digits, hyphens, or underscores.")

print("Input modality files:")
for modality in MODALITIES:
    path = SOURCE_FILES[modality]
    print(f"  {modality}: {path} ({path.stat().st_size / 1024**2:.1f} MiB)")
print(f"Checkpoint: {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size / 1024**3:.3f} GiB)")


## Verify the checkpoint and input geometry

The checkpoint is checked against the registry's recorded size and SHA-256,
then strictly loaded against the registered MONAI SwinUNETR. This validates
checkpoint compatibility, not training provenance, clinical performance, or
segmentation accuracy.


In [ ]:
import hashlib
import json
import shutil
import time
import psutil
import nibabel as nib
import numpy as np

from integrated_pipeline.modules.model_registry import load_model_registry
from src.data.brats import BraTSCase, load_input_volume
from src.segmentation3d.monai_engine import (
    predict_model_card_case,
    verify_model_card_checkpoint,
)

model_spec = load_model_registry()["segmentation_3d"]
manifest = model_spec["checkpoint_integrity"]
if CHECKPOINT_PATH.stat().st_size != manifest["size_bytes"]:
    raise ValueError("Checkpoint file size does not match the project's registered manifest.")
digest = hashlib.sha256()
with CHECKPOINT_PATH.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest().lower() != manifest["sha256"].lower():
    raise ValueError("Checkpoint SHA-256 does not match the project's registered manifest.")

case_dir = Path("/content/capstone_case") / CASE_ID
case_dir.mkdir(parents=True, exist_ok=False)
for modality, source in SOURCE_FILES.items():
    destination = case_dir / f"{CASE_ID}-{modality}.nii.gz"
    if source.name.endswith(".nii.gz"):
        shutil.copyfile(source, destination)
    else:
        nib.save(nib.load(str(source), mmap=True), str(destination))

case = BraTSCase(CASE_ID, CASE_ID.rsplit("-", 1)[0], case_dir)
volume = load_input_volume(case, modalities=MODALITIES, orientation=None)
reference_shape = tuple(int(value) for value in volume["image"].shape[1:])
reference_affine = np.asarray(volume["affine"], dtype=np.float64).copy()
assert tuple(model_spec["input_modalities"]) == MODALITIES
assert np.isfinite(volume["image"].numpy()).all()
del volume

compatibility = verify_model_card_checkpoint(CHECKPOINT_PATH, model_spec)
if compatibility["status"] != "compatible":
    raise RuntimeError(f"Checkpoint compatibility verification failed: {compatibility}")
print(json.dumps(compatibility, indent=2))
print(f"Four modalities passed project-loader checks: shape={reference_shape}")
print("Affine preserved from source NIfTI; no resampling/orientation conversion was applied.")


## Run inference

The existing project function performs sliding-window inference with batch size
1, the registry-configured 64³ ROI and overlap, sigmoid/threshold
postprocessing, NIfTI export, and visualization. `device="auto"` selects CUDA
when available and CPU otherwise. Sliding-window predictions are accumulated
on host memory to avoid storing a duplicate full input volume on GPU.


In [ ]:
from datetime import datetime, timezone
import shutil

OUTPUT_DIR = Path("/content/capstone_outputs") / (
    CASE_ID + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
before_ram = psutil.virtual_memory()
before_process = psutil.Process().memory_info().rss
started = time.perf_counter()
predict_model_card_case(
    CHECKPOINT_PATH,
    case_dir,
    OUTPUT_DIR,
    model_spec,
    device="auto",
)
wall_time_seconds = time.perf_counter() - started
after_ram = psutil.virtual_memory()
after_process = psutil.Process().memory_info().rss
metadata = json.loads((OUTPUT_DIR / "prediction_metadata.json").read_text(encoding="utf-8"))

print(f"Execution device: {metadata['device']}")
print(f"Inference wall time: {wall_time_seconds:.2f} s")
print(f"Engine inference time: {metadata['inference_time_seconds']:.2f} s")
print(f"System RAM total: {after_ram.total / 1024**3:.2f} GiB")
print(f"System RAM available before/after: {before_ram.available / 1024**3:.2f} / {after_ram.available / 1024**3:.2f} GiB")
print(f"Notebook process RSS before/after: {before_process / 1024**3:.2f} / {after_process / 1024**3:.2f} GiB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory telemetry: {json.dumps(metadata['memory_usage']['cuda_after'], indent=2)}")
print(f"CPU memory preflight: {json.dumps(metadata['memory_preflight'], indent=2)}")


## Validate masks and package downloadable outputs

The checks below compare each NIfTI mask to the source geometry, confirm
binary values, verify the expected ET/TC/WT output ordering and confirm the
visualization and inference metadata exist. No RC mask is expected or produced.


In [ ]:
assert metadata["modalities"] == ["t1n", "t1c", "t2w", "t2f"]
assert metadata["region_order"] == ["ET", "TC", "WT"]
assert "RC" in metadata["unavailable_regions"]
assert metadata["clinical_performance"] == "unvalidated"
assert metadata["inference"]["sw_batch_size"] == 1
assert metadata["output_shape"] == list(reference_shape)

for region in ("ET", "TC", "WT"):
    mask_path = OUTPUT_DIR / f"{CASE_ID}_{region}.nii.gz"
    mask_image = nib.load(str(mask_path))
    assert tuple(mask_image.shape) == reference_shape, (region, mask_image.shape, reference_shape)
    np.testing.assert_allclose(mask_image.affine, reference_affine, atol=1e-4)
    mask = np.asarray(mask_image.dataobj)
    assert np.isfinite(mask).all()
    assert set(np.unique(mask)).issubset({0, 1})
    assert region in metadata["probability_summary"]
    print(f"{region}: shape={mask_image.shape}, voxels={int(mask.sum())}, affine aligned")

visualization = OUTPUT_DIR / metadata["visualization"]
assert visualization.is_file() and visualization.stat().st_size > 0
assert (OUTPUT_DIR / "prediction_metadata.json").is_file()

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for item in OUTPUT_DIR.iterdir():
    shutil.copy2(item, DRIVE_OUTPUT_DIR / item.name)
ARCHIVE_PATH = Path("/content") / f"{CASE_ID}_SwinUNETR_ET_TC_WT_outputs.zip"
shutil.make_archive(str(ARCHIVE_PATH.with_suffix("")), "zip", OUTPUT_DIR)
print(f"Validated outputs copied to Drive: {DRIVE_OUTPUT_DIR}")
print(f"Download archive: {ARCHIVE_PATH}")


In [ ]:
from google.colab import files
files.download(str(ARCHIVE_PATH))


## Separate spatial-risk mapping stage (not part of segmentation)

The project keeps this as a distinct V8 stage. Its `run_v8_risk_mapping` adapter
requires **three real, spatially registered inputs**: the T1ce MRI, an actual
SynthSeg anatomical-label NIfTI, and the predicted WT tumor mask. It checks
shape and affine compatibility and rejects empty/non-integer/unknown
SynthSeg labels. ET/TC/WT masks are not substitutes for SynthSeg anatomy.
Never synthesize, infer, or relabel anatomical structures to satisfy this input.

The risk-map template is
`5_surgical_risk_mapping/configs/config_v7.json`; it contains the mapper's
configured weights and thresholds. The adapter creates a run-specific config
from that template and writes isolated outputs. It reports DTI/tract-dependent
components unavailable when those real registered data are absent. A risk map
or corridor output is a research prototype result, not a clinical risk score
or surgical recommendation.

After confirming that the SynthSeg label file is from this same study and has
the exact T1ce geometry, set the path below and opt in. The engine's own strict
registration/label checks remain authoritative; a mismatch must stop the run.


In [ ]:
RUN_SPATIAL_RISK_MAPPING = False
SYNTHSEG_PATH = Path("/content/drive/MyDrive/Capstone_MRI_Project/synthseg_labels.nii.gz")

if RUN_SPATIAL_RISK_MAPPING:
    if not SYNTHSEG_PATH.is_file():
        raise FileNotFoundError(f"Provide the real registered SynthSeg labels: {SYNTHSEG_PATH}")
    from integrated_pipeline.modules.risk_adapter import run_v8_risk_mapping

    risk_result = run_v8_risk_mapping(
        mri_path=case_dir / f"{CASE_ID}-t1c.nii.gz",
        tumor_mask_path=OUTPUT_DIR / f"{CASE_ID}_WT.nii.gz",
        synthseg_path=SYNTHSEG_PATH,
        output_dir=DRIVE_OUTPUT_DIR / "spatial_risk",
        patient_id=CASE_ID,
        config_template=PROJECT_ROOT / "5_surgical_risk_mapping/configs/config_v7.json",
    )
    print(f"Spatial-risk stage status: {risk_result['status']}")
    print(risk_result["note"])
else:
    print("Spatial-risk mapping not run. No SynthSeg labels or risk outputs were assumed.")
